# Retraining from scratch

We run these models separately from the main experiment so that, when a new unlearned model is made, these checkpoints can be quickly pulled, measured, and then set aside again.

### Imports

In [1]:
import sys
print(sys.version)

3.9.25 (main, Apr 17 2026, 00:00:00) 
[GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]


In [2]:
import os
import json

In [3]:
%ls

data/                              __pycache__/
evaluation/                        README.md
master_auditor.ipynb               results/
master_experiment.ipynb            trainer/
master_hyperparams.py              unlearn/
master_pretraining.ipynb           visualize_pretraining_results.ipynb
master_retrain_from_scratch.ipynb  visualize_results.ipynb
models/                            wandb/
_old/


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
# import matplotlib.pyplot as plt

# from trainer.utils import training_regimen_lr_annealing


### Set configs for the pretraining

In [5]:

from master_hyperparams import hyperparams
device = "cuda" if torch.cuda.is_available() else "cpu"


# ------- MAIN THINGS TO EDIT FOR THIS RUN ------- #
dataset = "CIFAR10"
model_class = "ResNet"
unlearning_type = "class"
# ------------------------------------------------- #

hp = hyperparams[dataset]
model_hp = hp[model_class]

retrain_config = {

    "description": "Retrain from Scratch - Resnet CIFAR10",
    
    "device": device,
    "model_class": model_class,
    "unlearning_type": unlearning_type,
    "data": {
        "dataset": dataset,
        "num_classes": hp["num_classes"],
        "batch_size": 1024,  # larger batch for faster pretraining
        "num_workers": hp["num_workers"],
        "item_to_unlearn": hp["items_to_unlearn"][unlearning_type]
        },

    "training": {
        "num_epochs": 75,
        "num_runs": 3,
        "learning_rate": model_hp["training"]["learning_rate"],
        "weight_decay": model_hp["training"]["weight_decay"],
        "batch_print_freq": 12,
        },
}


### Protocol for several runs

In [6]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /cs/student/msc/ml/2025/jmoncus/.netrc.
wandb: Currently logged in as: jjmoncus (jjmoncus706) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import glob
from models.archs.utils import init_model
from torch.optim.lr_scheduler import ReduceLROnPlateau
from trainer.utils import training_regimen_lr_annealing
from data.dataloaders import load_dataloaders_for_experiment
from evaluation.utils import measure_unlearning_metrics
import json
from data.utils import setup_seed
import time
from data.utils import split_forget_retain, split_random

def run_retrain_from_scratch(config, checkpoint_folder):
    
    print("="*70)
    print("="*19 + "  " + f'RUNNING RETRAINING FROM SCRATCH, SEED {config["GRAND_SEED"]}' + "  " + "="*19)
    print("="*70 + "\n")

    setup_seed(config["GRAND_SEED"])

    # create a subfolder for saving model checkpoints for this retraining
    print(f'All models will be of class {config["model_class"]}.\n')
    checkpoint_subfolder = os.path.join(checkpoint_folder, f"seed_{config['GRAND_SEED']}")
    if not os.path.exists(checkpoint_subfolder):   
        print(f"{checkpoint_subfolder} doesn't exist - creating it...\n")
        os.makedirs(checkpoint_subfolder, exist_ok=True)


    # Save the config for this retraining to the main checkpoints folder
    with open(os.path.join(checkpoint_subfolder, "retrain_from_scratch_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # ... decide what we're unlearning
    item_to_unlearn = config["data"]["item_to_unlearn"]

    # ... announce what we're unlearning
    retrain_name = f"{config['data']['dataset']}_{config['model_class']}_{config['training']['num_epochs']}_epochs_{config['unlearning_type']}_{item_to_unlearn}"
    print("-"*15 + "    " + retrain_name + "\n")
    

    # ... be intelligent about setting `class_to_replace` or `percent_to_replace` if either is None
    
    class_param = item_to_unlearn if config['unlearning_type'] == "class" else None
    percent_param = item_to_unlearn if config['unlearning_type'] == "percent" else None

    # ----------------------------------------------------------------------------------- #
    # ----------------------------- DEFINE UNLEARNING LOADERS --------------------------- #
    # ----------------------------------------------------------------------------------- #

    # ...  ------------- get some unlearning data for this config ------------------- #
    # ... the dataSET is fixed across runs, and the randomness within runs is handled by simply shuffling the data loader. There is no need to actually apply the micro-seed

    # only need `train`
    marked_train_loader, _, _ = load_dataloaders_for_experiment(
        name = config["data"]["dataset"],
        batch_size=config["data"]["batch_size"], 
        num_workers=config["data"]["num_workers"], 
        seed = config["GRAND_SEED"], 
        class_to_replace=class_param, 
        percent_to_replace=percent_param, 
        only_mark=True,
        val=False
        )
    # we make sure forget and retain sets are shuffled, to allow randomness across runs
    # only need `retain`
    print("Training - forget vs retain split:")
    _, retain_loader = split_forget_retain(marked_train_loader, batch_size=config["data"]["batch_size"], shuffle = True, num_workers=config["data"]["num_workers"])
    

    
    # ... and do a bunch of runs, where ...
    for i in range(1,  config["training"]["num_runs"]+1):

    
        # ----------------------------------------------------------------------------------- #
        # ----------------------------- RETRAIN FROM SCRATCH -------------------------------- #
        # ----------------------------------------------------------------------------------- #

        run_seed = config["GRAND_SEED"] * 1000 + i
        setup_seed(run_seed)
        
            # ... open new wandb session per run
        wandb.init(
            project="Verifying-Unlearning-2026",
            name=f"{run_seed}_retrain_{retrain_name}",
            config=config,
            reinit= "finish_previous"
            )
        
        print(f" ----- Retraining from scratch for run {i}, {retrain_name} ----- \n")
            
        # ... init a fresh model, opt, criterion, and scheduler for this run_seed
        empty_model = init_model(model_class = config["model_class"], num_classes = config['data']["num_classes"]).to(config["device"])
        opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
        criterion = nn.CrossEntropyLoss()
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            opt, 
            T_max=config["training"]["num_epochs"], 
            eta_min=1e-6
            )

        # ... do the training
        retrain_name = f"retrain_run_{i}_{retrain_name}"
        retrain_checkpoint_path = os.path.join(checkpoint_subfolder, f"{retrain_name}.pth")
        start = time.time() # EVENTUALLY NEEDS TO BE MEASURED SOME OTHER WAY
        retrained_model, opt, scheduler, retrain_retain_loss, retrain_retain_acc, retrain_retain_entr, retrain_retain_m_entr = training_regimen_lr_annealing(
            empty_model, 
            retain_loader,
            opt, 
            criterion, 
            scheduler, 
            device = config["device"], 
            num_epochs=config["training"]["num_epochs"], 
            model_path = retrain_checkpoint_path,
            print_freq = config["training"]["batch_print_freq"],
            w_and_b = True
            )
        end = time.time()
        wandb.log({"run time efficiency": end - start})
        

        # closes retrain wandb session
        wandb.finish()


    print("-"*75)
    print("-"*19 + "  " + f'FINISHED RETRAIN FROM SCRATCH, SEED {config["GRAND_SEED"]}' + "  " + "-"*19)
    print("-"*75 + "\n")
    

### Check metrics on unlearned models

In [8]:
# MAKE A RANDOM SEED
retrain_config["GRAND_SEED"] = 5
# DO EXP
run_retrain_from_scratch(config = retrain_config, checkpoint_folder="models/model_checkpoints/retrain_from_scratch")

===================  RUNNING RETRAINING FROM SCRATCH, SEED 5  ===================

setup random seed = 5
All models will be of class ResNet.

models/model_checkpoints/retrain_from_scratch/seed_5 doesn't exist - creating it...

---------------    CIFAR10_ResNet_75_epochs_class_5

========== DATALOADER INFO
Dataset: CIFAR-10
Train: 50000 images for training
Test: 10000 images for testing
Replaced class 5 in train
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize
num_workers = 4


Training - forget vs retain split:
Forget set: 5000 items
Retain set: 45000 items


setup random seed = 5001


 ----- Retraining from scratch for run 1, CIFAR10_ResNet_75_epochs_class_5 ----- 

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][11/44]	Loss 1.5527 (1.9610)	Accuracy 39.941 (28.638)	Entropy 1.6218 (1.7345)	M-Entropy 1.4451 (1.9183)	Time 6.37
Epoch: [1][23/44]	Loss 1.4411 (1.7375)	Accuracy 47.168 (35.657)	Entropy 1.4442 (1.6195)	M-Entropy 1.3817 (1.6813)	Time 2.39
Epoch: [1][35/44]	Loss 1.2656 (1.6117)	Accuracy 55.762 (40.492)	Entropy 1.2982 (1.5387)	M-Entropy 1.2290 (1.5563)	Time 2.39
train_accuracy (epoch) 42.844
Epoch 1 | LR: 1.0e-03 | RAM: 2.02GB | VRAM: 7.08GB | Weight Norm: 111.659
 ----- EPOCH 2 ----- 

Epoch: [2][11/44]	Loss 1.1930 (1.1843)	Accuracy 56.152 (56.315)	Entropy 1.1720 (1.1842)	M-Entropy 1.1783 (1.1684)	Time 3.16
Epoch: [2][23/44]	Loss 1.0897 (1.1536)	Accuracy 61.719 (57.890)	Entropy 1.0688 (1.1542)	M-Entropy 1.0894 (1.1415)	Time 2.39
Epoch: [2][35/44]	Loss 1.0112 (1.1149)	Accuracy 66.309 (59.603)	Entropy 0.9681 (1.1209)	M-Entropy 1

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


RAM_GB,▁▇▇▇████████████████████████████████▆▆▆▆
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇█
learning_rate,█████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
run time efficiency,▁
time (batch),▆▇▂▁▂▂▇▇▇▁▆▇▁▇▂▁▂▇▂▇▂▂▇▂▂▂▂█▂▂▂▇▂▂▁█▂▆▁▇
train_acc (batch),▁▃▅▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇▇█████████████████
train_acc (full),▁▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████████████████
train_entropy (batch),█▇▆▅▄▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_entropy (full),█▆▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


setup random seed = 5002


...

setup random seed = 5003


 ----- Retraining from scratch for run 3, CIFAR10_ResNet_75_epochs_class_5 ----- 

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][11/44]	Loss 1.6561 (1.9723)	Accuracy 37.598 (28.255)	Entropy 1.6104 (1.7675)	M-Entropy 1.5735 (1.9043)	Time 3.25
Epoch: [1][23/44]	Loss 1.4191 (1.7401)	Accuracy 48.340 (35.763)	Entropy 1.4570 (1.6346)	M-Entropy 1.3510 (1.6722)	Time 2.56
Epoch: [1][35/44]	Loss 1.3242 (1.6248)	Accuracy 49.316 (39.912)	Entropy 1.3454 (1.5569)	M-Entropy 1.2705 (1.5612)	Time 2.42
train_accuracy (epoch) 42.069
Epoch 1 | LR: 1.0e-03 | RAM: 2.10GB | VRAM: 7.09GB | Weight Norm: 111.797
 ----- EPOCH 2 ----- 

Epoch: [2][11/44]	Loss 1.1751 (1.2125)	Accuracy 58.203 (55.094)	Entropy 1.1771 (1.2356)	M-Entropy 1.1575 (1.1821)	Time 3.19
Epoch: [2][23/44]	Loss 1.0627 (1.1704)	Accuracy 63.281 (57.243)	Entropy 1.0732 (1.1834)	M-Entropy 1.0534 (1.1495)	Time 2.46
Epoch: [2][35/44]	Loss 0.9853 (1.1303)	Accuracy 63.770 (58.735)	Entropy 1.0530 (1.1436)	M-Entropy 0

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


RAM_GB,█████████████████████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
learning_rate,██████▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
run time efficiency,▁
time (batch),▁▁▁▁▇▁▇▁▁▂▁▇▂▂▂▂█▁▂▇▇▇▇▇▁▂▇▇▁▁▁▇▂▃▁▁█▂▂▇
train_acc (batch),▁▃▄▅▅▆▆▆▇▆▇▇▇▇▇█▇▇▇▇████████████████████
train_acc (full),▁▃▃▄▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇████████████████████
train_entropy (batch),█▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_entropy (full),█▆▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


---------------------------------------------------------------------------
-------------------  FINISHED RETRAIN FROM SCRATCH, SEED 5  -------------------
---------------------------------------------------------------------------

